In [1]:
%matplotlib qt5

import hyperspy.api as hs
import pyxem as pxm
import numpy as np
import orix
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import matplotlib as mpl
import pickle
import copy

from pathlib import Path
from diffpy.structure import Atom, Lattice, Structure
from diffsims.generators.simulation_generator import SimulationGenerator
from dataclasses import dataclass
from sklearn.decomposition import PCA

from orix.quaternion import symmetry, OrientationRegion
from orix.vector import Vector3d, Miller
from orix.plot import IPFColorKeyTSL

c:\Users\denis\hyperspy-bundle_latest\Lib\site-packages\hyperspy\misc\utils.py:72: VisibleDeprecationWarning: `_get_block_pattern` has moved to `hyperspy.misc.dask_utils`. It is for internal use only and may be removed in the future.
  warnings.warn(


In [2]:
## Paths used for loading and saving data
path = Path("D:\\Datasets\\aluminium_tilt_series\\SPED_tilt_series")
path_calibrated = Path("D:\\Datasets\\aluminium_tilt_series\\series_calibrated")

path_target = Path("D:\\Datasets\\aluminium_tilt_series\\target_areas")
path_reference = Path("D:\\Datasets\\aluminium_tilt_series\\reference_areas")

path_target_mean = Path("D:\\Datasets\\aluminium_tilt_series\\target_mean")
path_reference_mean = Path("D:\\Datasets\\aluminium_tilt_series\\reference_mean")

path_target_sum = Path("D:\\Datasets\\aluminium_tilt_series\\target_sum")
path_reference_sum = Path("D:\\Datasets\\aluminium_tilt_series\\reference_sum")

path_target_masked = Path("D:\\Datasets\\aluminium_tilt_series\\target_areas_masked")
path_reference_masked = Path("D:\\Datasets\\aluminium_tilt_series\\reference_areas_masked")

path_target_template_disk = Path("D:\\Datasets\\aluminium_tilt_series\\target_template_disk")
path_reference_template_disk = Path("D:\\Datasets\\aluminium_tilt_series\\reference_template_disk")

raw_files = sorted(path.glob("*.hspy"), key=lambda x: int(x.stem.split("_")[-1]))
calibrated_files = sorted(path_calibrated.glob("*.hspy"), key=lambda x: int(x.stem.split("_")[-1]))

target_files = sorted(path_target.glob("*.hspy"), key=lambda x: int(x.stem.split("_")[-1]))
reference_files = sorted(path_reference.glob("*.hspy"), key=lambda x: int(x.stem.split("_")[-1]))

target_files_mean = sorted(path_target_mean.glob("*.hspy"), key=lambda x: int(x.stem.split("_")[-1]))
reference_files_mean = sorted(path_reference_mean.glob("*.hspy"), key=lambda x: int(x.stem.split("_")[-1]))

target_files_sum = sorted(path_target_sum.glob("*.hspy"), key=lambda x: int(x.stem.split("_")[-1]))
reference_files_sum = sorted(path_reference_sum.glob("*.hspy"), key=lambda x: int(x.stem.split("_")[-1]))

target_files_masked = sorted(path_target_masked.glob("*.hspy"), key=lambda x: int(x.stem.split("_")[-1]))
reference_files_masked = sorted(path_reference_masked.glob("*.hspy"), key=lambda x: int(x.stem.split("_")[-1]))

target_files_template_disk = sorted(path_target_template_disk.glob("*.hspy"), key=lambda x: int(x.stem.split("_")[-1]))
reference_files_template_disk = sorted(path_reference_template_disk.glob("*.hspy"), key=lambda x: int(x.stem.split("_")[-1]))

## Preprocessing

In [ ]:
# The scaling factor used was determined by manually calibrating one of the datasets and extracting the scale factor from the calibrated data. #
def preprocess_data(path, path_calibrated, path_target, path_reference):
    files = sorted(path.glob("*.hspy"))
    for j, i in enumerate(files):
        filename = i.stem
        number = filename.rsplit("_", 1)[-1]
        ## Calibrate the data ##
        load_i = hs.load(i)
        
        if j == 0:
            load_i.calibrate()
            plt.show(block=True)
            scale_factor = load_i.axes_manager[2].scale

        load_i.axes_manager[2].scale = scale_factor
        load_i.axes_manager[3].scale = scale_factor

        load_i.axes_manager[2].units = "1/Å"
        load_i.axes_manager[3].units = "1/Å"

        shifts = load_i.get_direct_beam_position(method="blur", sigma=5, half_square_width=10)
        linear_shifts = shifts.get_linear_plane()
        load_i_centered = load_i.center_direct_beam(shifts=linear_shifts, inplace=False)


        ## Select region of interest ##
        reference_rectangle = hs.roi.RectangularROI(left=0, top=0, right=10, bottom=10)
        target_rectangle = hs.roi.RectangularROI(left=0, top=0, right=10, bottom=10)

        load_i_centered.plot()

        reference_roi = reference_rectangle.interactive(load_i_centered)
        target_roi = target_rectangle.interactive(load_i_centered, color='C1')
        
        plt.show(block=True)

        reference_roi.save(path_reference / f"SPED_tilt_series_reference_{number}.hspy")
        target_roi.save(path_target / f"SPED_tilt_series_target_{number}.hspy")
        load_i_centered.save(path_calibrated / f"SPED_tilt_series_calibrated_{number}.hspy")

### Masking

In [ ]:
def masking_function(path_target, path_reference, path_target_masked, path_reference_masked, path_target_mean, path_reference_mean):
    files_target = sorted(path_target.glob("*.hspy"))
    files_reference = sorted(path_reference.glob("*.hspy"))

    for number, j in enumerate(zip(files_target, files_reference)):
        
        target = hs.load(j[0])
        reference = hs.load(j[1])

        target_mean = target.mean()
        reference_mean = reference.mean()
        
        target_mean = target_mean.data[None, None, :, :]
        reference_mean = reference_mean.data[None, None, :, :]

        target_mean = hs.signals.Signal2D(target_mean)
        target_mean.set_signal_type("electron_diffraction")

        reference_mean = hs.signals.Signal2D(reference_mean)
        reference_mean.set_signal_type("electron_diffraction")

        st_target = target_mean.template_match_disk(disk_r=7, subtract_min=False)
        st_reference = reference_mean.template_match_disk(disk_r=7, subtract_min=False)

        peaks_target = st_target.get_diffraction_vectors(min_distance=15, threshold_abs=0.25)
        peaks_reference = st_reference.get_diffraction_vectors(min_distance=15, threshold_abs=0.25)

        mask_target = peaks_target.to_mask(disk_r = 10)
        mask_reference = peaks_reference.to_mask(disk_r = 10)

        center_beam_target = ~st_target.get_direct_beam_mask(10)
        center_beam_reference = ~st_reference.get_direct_beam_mask(10)

        masked_target = target_mean * mask_target
        masked_target = masked_target * center_beam_target

        masked_reference = reference_mean * mask_reference
        masked_reference = masked_reference * center_beam_reference

        masked_target = masked_target
        masked_reference = masked_reference

        ny, nx = target.axes_manager.signal_shape
        center = (ny/2 - 0.5, nx/2 - 0.5)

        def axes_calibrate(signal, target, center):
            signal.axes_manager[2].scale = target.axes_manager[2].scale
            signal.axes_manager[3].scale = target.axes_manager[3].scale
            signal.axes_manager[2].units = target.axes_manager[2].units
            signal.axes_manager[3].units = target.axes_manager[3].units

            signal.calibration(center=center)

        axes_calibrate(masked_target, target, center)
        axes_calibrate(masked_reference, reference, center)
        axes_calibrate(target_mean, target, center)
        axes_calibrate(reference_mean, reference, center)

        axes_calibrate(st_target, target, center)
        axes_calibrate(st_reference, reference, center)

        masked_target.save(path_target_masked / f"target_masked_{number}.hspy", overwrite=True)
        masked_reference.save(path_reference_masked / f"reference_masked_{number}.hspy", overwrite=True)

        target_mean.save(path_target_mean / f"target_mean_{number}.hspy", overwrite=True)
        reference_mean.save(path_reference_mean / f"reference_mean_{number}.hspy", overwrite=True)

        st_target.save(path_target_template_disk / f"target_template_disk_{number}.hspy", overwrite=True)
        st_reference.save(path_reference_template_disk / f"reference_template_disk_{number}.hspy", overwrite=True)


masking_function(path_target, path_reference, path_target_masked, path_reference_masked, path_target_mean, path_reference_mean)

## Simulation

In [3]:
# Define the crystal structure for aluminum (Al)
a = 4.05
atoms = [Atom('Al', [0, 0, 0]), Atom('Al', [0.5, 0.5, 0]), Atom('Al', [0.5, 0, 0.5]), Atom('Al', [0, 0.5, 0.5])]
lattice = Lattice(a, a, a, 90, 90, 90)
phase = orix.crystal_map.Phase(name='Al', space_group=225, structure=Structure(atoms, lattice))

angular_resolution = 0.5
grid = orix.sampling.get_sample_reduced_fundamental(angular_resolution, point_group=phase.point_group)
orientations = orix.quaternion.Orientation(grid, symmetry=phase.point_group)

scaling_factor = hs.load(reference_files[0]).axes_manager[2].scale
rec_radius = 128 * scaling_factor
simgen = SimulationGenerator(precession_angle=1., minimum_intensity=1e-5)
simulations = simgen.calculate_diffraction2d(phase=phase, rotation=grid, reciprocal_radius=rec_radius, with_direct_beam=False, max_excitation_error=0.01)

# Create IPF color keys for x, y, and z directions
key_x = IPFColorKeyTSL(phase.point_group, Vector3d.xvector())
key_y = IPFColorKeyTSL(phase.point_group, Vector3d.yvector())
key_z = IPFColorKeyTSL(phase.point_group, Vector3d.zvector())

In [4]:
@dataclass
class SPEDResult:
    data: object # Indices, correlations, orientations
    orientations: object
    results : object = None

    def load_results(self):
        s = hs.signals.Signal2D(self.data)

        s.axes_manager.navigation_axes[0].name = "y"
        s.axes_manager.navigation_axes[1].name = "x"

        s.axes_manager.signal_axes[0].name = "n-best"
        s.axes_manager.signal_axes[1].name = "columns"

        s.set_signal_type("orientation_map")
        s.simulation = simulations
        s.column_names = ["index", "correlation", "rotation", "factor"]
        s.units = ["a.u.", "a.u.", "deg", "a.u."]

        self.results = s

    def plot_result(self, dp_image_path=None):        
        fig = plt.figure()
        ax = fig.add_subplot(111, projection="ipf", symmetry=phase.point_group)

        template_index = self.data[0, 0, 0, 0].astype("int16")
        optical_axis = Miller(uvw=[0, 0, 1], phase=phase)
        miller = (simulations.get_simulation(template_index)[0] * optical_axis).round()

        cors = self.data[0, 0, :, 1] / np.max(self.data[0, 0, :, 1])
        ax.scatter(self.orientations[0, 0], c=cors, cmap='magma')
        ax.scatter(self.orientations[0, 0, 0], c="r", marker="x", label=f"Best match: {miller.uvw[0]}")
        
        if dp_image_path is not None:
            dp_image = hs.load(dp_image_path)

            dp_image.plot(cmap='viridis', norm='symlog')
            dp_image.add_marker(self.results.to_markers(annotate=True))
        ax.legend()
        plt.show()

In [ ]:
def template(target_masked, ref_masked):
    target_masked.change_dtype("float32")
    ref_masked.change_dtype("float32")

    target_azi = target_masked.get_azimuthal_integral2d(npt=256)
    ref_azi = ref_masked.get_azimuthal_integral2d(npt=256)
    
    target_results = target_azi.get_orientation(simulations, n_best=grid.size, frac_keep=0.5)
    ref_results = ref_azi.get_orientation(simulations, n_best=grid.size, frac_keep=0.5)
    
    target_orientations = target_results.to_single_phase_orientations()
    reference_orientations = ref_results.to_single_phase_orientations()
    
    target = SPEDResult(
        data=target_results.data,
        orientations=target_orientations)
    
    reference = SPEDResult(
        data=ref_results.data,  
        orientations=reference_orientations)
    
    return target, reference

In [ ]:
def tilt_results(target_set, reference_set):
    misorientation_array = np.empty((len(target_set), 2), dtype=object)

    for i, files in enumerate(zip(target_set, reference_set)):
        
        target_data = hs.load(files[0])
        reference_data = hs.load(files[1])

        target_result, reference_result = template(target_data, reference_data)

        misorientation_array[i,0] = target_result
        misorientation_array[i,1] = reference_result
    
    return misorientation_array

result_array = tilt_results(target_files_masked, reference_files_masked)
pickle.dump(result_array, open("result_array_masked.pkl", "wb"))

In [6]:
result_path_masked = Path("C:\\Users\\Denis\\Documents\\GitHub\\Masteroppgave") / "result_array_masked.pkl"

with open(result_path_masked, "rb") as f:
    result_masked_array = pickle.load(f)

In [8]:
result_masked_array = result_masked_array[10:, :]

In [9]:
def return_orix_orientation(result_array):
    orientation_target = np.empty(50, dtype=object)
    orientation_reference = np.empty(50, dtype=object)

    for number, i in enumerate(result_array):
        orientation_target[number] = i[0].orientations[0, 0]
        orientation_reference[number] = i[1].orientations[0, 0]

    quat_array = np.stack([j.data for j in orientation_target])
    orientation_target = orix.quaternion.Orientation(quat_array, symmetry=phase.point_group)

    quat_array = np.stack([k.data for k in orientation_reference])
    orientation_reference = orix.quaternion.Orientation(quat_array, symmetry=phase.point_group)

    return orientation_target, orientation_reference

orientation_target, orientation_reference = return_orix_orientation(result_masked_array)
ax_target, ax_reference = orientation_target.to_axes_angles(), orientation_reference.to_axes_angles()

In [10]:
def fit_line_pca(points):
    centroid = points.mean(axis=0)
    pca = PCA(n_components=1)
    pca.fit(points)
    direction = pca.components_[0]
    direction = direction / np.linalg.norm(direction)
    return centroid, direction

def point_line_dist(points, point_on_line, direction):
    diff = points - point_on_line
    return np.linalg.norm(np.cross(diff, direction), axis=1)

def ransac_line_fit(axis_data, n_iter=2000, threshold=0.05, min_inliers=10):
    data_xyz = np.array(axis_data.xyz).transpose()[0, :, :]

    n = len(data_xyz)
    best_inliers = None
    best_count = 0

    for _ in range(n_iter):
        idx = np.random.choice(n, 2, replace=False)
        p1, p2 = data_xyz[idx]

        direction = p2 - p1
        norm = np.linalg.norm(direction)
        if norm < 1e-12:
            continue
        direction = direction / norm

        dist = point_line_dist(data_xyz, p1, direction)
        inliers = dist < threshold
        count = np.sum(inliers)

        if count > best_count:
            best_count = count
            best_inliers = inliers

    if best_inliers is None or best_count < min_inliers:
        raise ValueError("No good line found. Try increasing threshold or n_iter.")

    inlier_points = data_xyz[best_inliers]
    centroid, direction = fit_line_pca(inlier_points)

    t = np.linspace(-0.7, 0.4, 60)
    line = centroid + t[:, None] * direction

    return centroid, direction, inlier_points, best_inliers, line

In [14]:
centroid, direction, inlier_points_target, inlier_mask_target, line = ransac_line_fit(
    ax_target,
    n_iter=4000,
    threshold=0.10,
    min_inliers=5
)

centroid2, direction2, inlier_points_reference, inlier_mask_reference, line2 = ransac_line_fit(
    ax_reference,
    n_iter=4000,
    threshold=0.10,
    min_inliers=5
)

In [15]:
def expected_t_from_local_steps(inline_idx, line_points, query_idx):

    inline_idx = np.asarray(inline_idx)
    line_points = np.asarray(line_points)
    query_idx = np.asarray(query_idx)

    if len(inline_idx) < 2:
        raise ValueError("Need at least two inline points.")

    idx_diff = np.diff(inline_idx)
    line_diff = np.diff(line_points)
    steps = line_diff / idx_diff   # local step per index in each interval

    t_expected = np.empty(len(query_idx), dtype=float)

    for n, q in enumerate(query_idx):
        pos = np.searchsorted(inline_idx, q)

        # Case 1: q is before the first inline point
        if pos == 0:
            step = steps[0]
            t_expected[n] = line_points[0] + step * (q - inline_idx[0])

        # Case 2: q is after the last inline point
        elif pos == len(inline_idx):
            step = steps[-1]
            t_expected[n] = line_points[-1] + step * (q - inline_idx[-1])

        # Case 3: q lands exactly on an inline point
        elif inline_idx[pos] == q:
            t_expected[n] = line_points[pos]

        # Case 4: q lies between two inline points
        else:
            left_idx = inline_idx[pos - 1]
            t_left = line_points[pos - 1]
            step = steps[pos - 1]
            t_expected[n] = t_left + step * (q - left_idx)

    return t_expected

In [16]:
def choose_candidates_smooth(
    candidates,
    centroid,
    direction,
    inliers,
    inlier_mask,
):
    data = np.array(candidates.xyz).transpose(1, 0, 2)
    inlier_idx = np.where(inlier_mask)[0]

    if len(inlier_idx) < 2:
        raise ValueError("Need at least two inliers to estimate expected positions.")

    # t-values of trusted inlier points
    line_points = np.dot(inliers - centroid, direction)

    # expected t for every sequence index
    all_idx = np.arange(data.shape[0])
    t_expected = expected_t_from_local_steps(inlier_idx, line_points, all_idx)

    expected_points = centroid + np.outer(t_expected, direction)

    best_match = np.argmin(np.linalg.norm(data[~inlier_mask, :, :] - expected_points[~inlier_mask, :, None], axis=1), axis=1)
    
    return best_match

best_match_target = choose_candidates_smooth(  
    ax_target, centroid, direction, inlier_points_target, inlier_mask_target)

best_match_reference = choose_candidates_smooth(     
    ax_reference, centroid2, direction2, inlier_points_reference, inlier_mask_reference)


In [17]:
def plot_line_fit(data, inliers, fixed_points, line):

    fig = plt.figure(figsize=(12, 6))
    ax = fig.add_subplot(111, projection='axangle')
    fundamental_zone = OrientationRegion.from_symmetry(data.symmetry)

    # --- all points ---
    ax.scatter(data[:, 0], alpha=0.3, s=10, label="All points")
    
    # --- inliers ---
    ax.scatter(inliers[:, 0], s=10, label="Inliers")
    
    # --- fixed points ---
    ax.scatter(fixed_points[:, 0], s=10, label="Fixed points")
    
    # --- fitted line ---
    ax.scatter(line, linewidth=3, label="Fitted line")
    
    # --- fundamental zone ---
    ax.plot_wireframe(fundamental_zone, color='k', alpha=0.5)

    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_zlabel("z")
    
    ax.legend()
    plt.tight_layout()
    plt.show()


In [18]:
def plot_ipf(orientation, line=None):
    c_scalar = np.arange(1, orientation.shape[0] + 1)

    directions = Vector3d([[1, 0, 0], [0, 1, 0], [0, 0, 1]])
    titles = ["x", "y", "z"]

    fig = plt.figure(figsize=(15, 6))
    plt.rcParams["axes.grid"] = True

    # Create colorbar (shared)
    norm = mpl.colors.Normalize(vmin=c_scalar.min(), vmax=c_scalar.max())
    cmap = plt.cm.get_cmap('viridis')
    sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])

    colors = cmap(norm(c_scalar))

    axes = []
    for i, d in enumerate(directions):
        ax = fig.add_subplot(1, 3, i + 1, projection='ipf', direction=d, symmetry=phase.point_group)
        axes.append(ax)
        if line is not None:
            ax.scatter(line, c='k', linewidth=0.4)

        ax.scatter(orientation, c=colors)
        ax.set_title(titles[i])

    # Add colorbar in its own axis (right side)
    cax = fig.add_axes([0.93, 0.15, 0.02, 0.7])  # [left, bottom, width, height]
    cbar = fig.colorbar(sm, cax=cax)
    cbar.set_label("Dataset number")
    
    plt.subplots_adjust(left=0.05, right=0.9, wspace=0.25)

In [19]:
def axis_angle_array_to_quaternion(R):
    theta = np.linalg.norm(R, axis=1)
    
    q = np.zeros((len(R), 4))
    
    small = theta < 1e-12
    large = ~small
    
    # small angles
    q[small, 0] = 1.0
    
    # normal case
    n = R[large] / theta[large][:, None]
    
    q[large, 0] = np.cos(theta[large] / 2)
    q[large, 1:] = n * np.sin(theta[large] / 2)[:, None]
    
    return q

orix_line_target = orix.quaternion.Orientation(axis_angle_array_to_quaternion(line), symmetry=phase.point_group)
orix_line_reference = orix.quaternion.Orientation(axis_angle_array_to_quaternion(line2), symmetry=phase.point_group)

fixed_orientations = orientation_target[:]
fixed_orientations[~inlier_mask_target, 0] = orientation_target[~inlier_mask_target, best_match_target]

fixed_orientations2 = orientation_reference[:]
fixed_orientations2[~inlier_mask_reference, 0] = orientation_reference[~inlier_mask_reference, best_match_reference]

# Visualization

In [ ]:
plt.rcParams["axes.grid"] = True

line_vector = orix_line_target * Vector3d.zvector()
line2_vector = orix_line_reference * Vector3d.zvector()

vectors_target = fixed_orientations[:, 0] * Vector3d.zvector()
vectors_reference = fixed_orientations2[:, 0] * Vector3d.zvector()

cross_target = line_vector[-10].cross(line_vector[0])
cross_reference = line2_vector[-10].cross(line2_vector[10])

cross = cross_target.cross(cross_reference)

fig6 = vectors_target.scatter(
    s=5,
    axes_labels=["RD", "TD", None],
    show_hemisphere_label=True,
    return_figure=True,
)

vectors_reference.scatter(
    s=5,
    figure=fig6,
    c='C1'
)

cross_target.scatter(
    s=50,
    figure=fig6
)

cross_reference.scatter(
    s=50,
    c='C1',
    figure=fig6
)
cross_target.draw_circle(
    figure=fig6)

cross_reference.draw_circle(
    c='C1',
    figure=fig6)

cross.scatter(
    s=100,
    c='k',
    figure=fig6
)

In [20]:
plot_line_fit(orientation_target, orientation_target[inlier_mask_target], fixed_orientations, orix_line_target)
plot_line_fit(orientation_reference, orientation_reference[inlier_mask_reference], fixed_orientations2, orix_line_reference)

In [21]:
plot_ipf(orientation_target[:, 0], orix_line_target)
plot_ipf(fixed_orientations[:, 0], orix_line_target)

C:\Users\denis\AppData\Local\Temp\ipykernel_30208\3883246036.py:12: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = plt.cm.get_cmap('viridis')


In [22]:
plot_ipf(orientation_reference[:, 0], orix_line_reference)
plot_ipf(fixed_orientations2[:, 0], orix_line_reference)

C:\Users\denis\AppData\Local\Temp\ipykernel_30208\3883246036.py:12: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = plt.cm.get_cmap('viridis')


In [23]:
def correction_swapping():
    swapped = copy.deepcopy(result_masked_array)
    index_target = np.where(~inlier_mask_target)[0]
    index_reference = np.where(~inlier_mask_reference)[0]
    for n, i in enumerate(index_target):
        swapped[i, 0].data[0, 0, 0, :] = swapped[i, 0].data[0, 0, best_match_target[n], :]
        swapped[i, 0].orientations[0, 0, 0] = swapped[i, 0].orientations[0, 0, best_match_target[n]]
    for k, j in enumerate(index_reference):
        swapped[j, 1].data[0, 0, 0, :] = swapped[j, 1].data[0, 0, best_match_reference[k], :]
        swapped[j, 1].orientations[0, 0, 0] = swapped[j, 1].orientations[0, 0, best_match_reference[k]]
    
    return swapped

swapped_results = correction_swapping()

In [24]:
def correction_dp_comparison(index, target_or_reference):
    if target_or_reference == "target":
        swapped_results[index, 0].load_results()
        swapped_results[index, 0].plot_result(target_files_mean[index])
        result_masked_array[index, 0].load_results()
        result_masked_array[index, 0].plot_result(target_files_mean[index])
    elif target_or_reference == "reference":
        swapped_results[index, 1].load_results()
        swapped_results[index, 1].plot_result(reference_files_mean[index])
        result_masked_array[index, 1].load_results()
        result_masked_array[index, 1].plot_result(reference_files_mean[index])

In [44]:
correction_dp_comparison(30, "reference")

In [28]:
misorientation = orientation_target[:, 0].angle_with(orientation_reference[:, 0])
mean_misorientation = np.mean(misorientation)

misorientation_fixed = fixed_orientations[:, 0].angle_with(fixed_orientations2[:, 0])
mean_misorientation_fixed = np.mean(misorientation_fixed)

number = np.arange(10, 60)

fig = plt.figure()
plt.plot(number, misorientation*180/np.pi, marker='o', label='Misorientation')
plt.axhline(mean_misorientation*180/np.pi, color='r', linestyle='--', label='Mean Misorientation')
plt.plot(number, misorientation_fixed*180/np.pi, marker='o', label='Misorientation (fixed)')
plt.axhline(mean_misorientation_fixed*180/np.pi, color='g', linestyle='--', label='Mean Misorientation (fixed)')
plt.axhline(50.38, color='k', linestyle='-', label='Expected Misorientation')
plt.title('Misorientation vs Dataset Number')
plt.xlabel('Dataset Number')
plt.ylabel('Misorientation (degrees)')
plt.legend()